🎓 Projet M2 : Graphe CV–Job & Prédiction Hybride

1. Configuration et Données

Nous installons les outils pour la manipulation de graphes (networkx), le traitement du langage naturel (sentence-transformers) et l'apprentissage automatique (scikit-learn).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
from networkx.algorithms import bipartite, community
from sentence_transformers import SentenceTransformer, util
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score ,confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from itertools import combinations

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

model = SentenceTransformer('all-MiniLM-L6-v2')

2. Construction et Modélisation du Graphe Biparti

Nous créons ici les nœuds CV et Job avec leurs attributs respectifs (compétences, niveaux, domaines).

In [ ]:
G = nx.Graph()
domaines = {
    "Data": ["Python", "Machine Learning", "SQL", "Pandas", "Scikit-learn", "Deep Learning", "Tableau"],
    "Web": ["JavaScript", "React", "Node.js", "HTML", "CSS", "TypeScript", "Next.js"],
    "DevOps": ["Docker", "Kubernetes", "AWS", "CI/CD", "Linux", "Terraform", "Jenkins"],
    "Marketing": ["SEO", "Google Analytics", "Content Strategy", "Social Media", "Copywriting", "Ads"]
}

niveaux = ["junior", "intermédiaire", "senior"]

# Génération de 20 CV
for i in range(20):
    dom = random.choice(list(domaines.keys()))
    niv = random.choice(niveaux)
    skills = random.sample(domaines[dom], k=random.randint(3, 5))
    text = f"{', '.join(skills)}. {niv}."
    G.add_node(f"CV_Gen_{i}", type="CV", text=text, label=niv, bipartite=0)

# Génération de 10 Jobs
for i in range(10):
    dom = random.choice(list(domaines.keys()))
    req_skills = random.sample(domaines[dom], k=2)
    text_job = f"Expert en {', '.join(req_skills)}"
    G.add_node(f"Job_Gen_{i}", type="Job", text=text_job, bipartite=1)

cv_nodes = [n for n, d in G.nodes(data=True) if d['type'] == 'CV']
job_nodes = [n for n, d in G.nodes(data=True) if d['type'] == 'Job']

for cv in cv_nodes:
    targets = random.sample(job_nodes, k=random.randint(1, 2))
    for t in targets:
        G.add_edge(cv, t)

print(f"Graphe enrichi : {G.number_of_nodes()} nœuds et {G.number_of_edges()} arêtes.")

In [ ]:
def add_internal_cv_features(G):
    for n, d in G.nodes(data=True):
        if d.get('type') == 'CV':
            # On sépare le texte pour identifier les compétences (ex: par virgule)
            skills = [s.strip().lower() for s in d['text'].split(',')]
            internal_g = nx.Graph()
            # On lie les compétences qui apparaissent ensemble dans le même CV
            internal_g.add_edges_from(combinations(skills, 2))

            # Feature demandée : densité du graphe de compétences interne
            G.nodes[n]['internal_density'] = nx.density(internal_g) if len(internal_g) > 1 else 0
    return G

G = add_internal_cv_features(G)
print("Caractéristiques des communautés internes calculées.")

3. Analyse Structurelle et Communautés Internes

Nous analysons la structure et extrayons des "communautés de compétences" au sein des CV.

In [ ]:
print(f"Degrés : {dict(G.degree())}")
print(f"Densité : {nx.density(G)}")

communities = community.louvain_communities(G, seed=SEED)
for i, comm in enumerate(communities):
    for node in comm:
        G.nodes[node]['community_global'] = i

4. Prédiction de Liens Hybride (LLM + Algorithmes)

C'est ici que nous combinons l'analyse sémantique (LLM) et la topologie du graphe.

In [ ]:
def get_llm_similarity(node1, node2):
    emb1 = model.encode(G.nodes[node1]['text'], convert_to_tensor=True)
    emb2 = model.encode(G.nodes[node2]['text'], convert_to_tensor=True)
    return util.pytorch_cos_sim(emb1, emb2).item()

def structural_link_prediction(graph):
    cv_nodes = [n for n, d in graph.nodes(data=True) if d['type'] == 'CV']
    projection = bipartite.projected_graph(graph, cv_nodes)
    return list(nx.adamic_adar_index(projection))

top_nodes = {n for n, d in G.nodes(data=True) if d['bipartite'] == 0}
def jaccard_bipartite(u, v):
    u_neighbors = set(G.neighbors(u))
    v_neighbors = set(G.neighbors(v))
    if not u_neighbors or not v_neighbors: return 0
    return len(u_neighbors & v_neighbors) / len(u_neighbors | v_neighbors)

new_links = []
for cv in [n for n, d in G.nodes(data=True) if d['type'] == 'CV']:
    for job in [n for n, d in G.nodes(data=True) if d['type'] == 'Job']:
        if not G.has_edge(cv, job):
            score = get_llm_similarity(cv, job)
            if score > 0.75:
                new_links.append((cv, job, score))

G.add_weighted_edges_from(new_links, label='predicted')

In [ ]:
def get_classical_predictions(G):
    cv_nodes = [n for n, d in G.nodes(data=True) if d['bipartite'] == 0]
    projection = bipartite.projected_graph(G, cv_nodes)

    preds = nx.adamic_adar_index(projection)
    return sorted(preds, key=lambda x: x[2], reverse=True)

classical_results = get_classical_predictions(G)
print("Top prédictions structurelles (Adamic-Adar) générées.")

5. Représentation des Nœuds et Classification

Nous construisons des vecteurs de caractéristiques (features) pour entraîner un classifieur.

In [ ]:
X = []
y_text = []

cv_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'CV']

for n in cv_nodes:
    d = G.nodes[n]

    emb = model.encode(d['text'])

    graph_feats = [
        G.degree(n),
        d.get('community_global', 0),
        d.get('internal_density', 0)
    ]

    X.append(np.concatenate([emb, graph_feats]))
    y_text.append(d['label'])

X = np.array(X)
y_text = np.array(y_text)

le = LabelEncoder()
y_encoded = le.fit_transform(y_text)

clf = RandomForestClassifier(random_state=42)
clf.fit(X, y_encoded)
predictions_encoded = clf.predict(X)

print("\n--- Évaluation Détaillée (Dataset Complet) ---")
y_true_names = le.inverse_transform(y_encoded)
y_pred_names = le.inverse_transform(predictions_encoded)

print(classification_report(y_true_names, y_pred_names, target_names=le.classes_))

cm = confusion_matrix(y_true_names, y_pred_names, labels=le.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title(f"Matrice de Confusion ({len(cv_nodes)} CV analysés)")
plt.show()

In [ ]:
def plot_bipartite_communities(graph):
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(graph, k=0.5, seed=SEED)

    colors = ['skyblue' if graph.nodes[n]['type'] == 'CV' else 'orange' for n in graph.nodes()]

    node_sizes = [v * 500 for v in nx.degree_centrality(graph).values()]

    nx.draw(graph, pos, with_labels=True, node_color=colors,
            node_size=node_sizes, edge_color='gray', alpha=0.7)

    plt.title("Visualisation du Graphe CV-Job : Centralité et Type")
    plt.show()

plot_bipartite_communities(G)

In [ ]:
print("--- Analyse des Profils Atypiques (Outliers) ---")

betweenness = nx.betweenness_centrality(G)

bet_values = [v for n, v in betweenness.items() if G.nodes[n]['type'] == 'CV']
threshold = np.mean(bet_values) + np.std(bet_values)

outliers = []
for n, score in betweenness.items():
    if G.nodes[n]['type'] == 'CV' and score > threshold:
        outliers.append((n, score, G.nodes[n].get('text', 'N/A')))

if not outliers:
    outliers = sorted([(n, betweenness[n], G.nodes[n]['text'])
                       for n in G.nodes if G.nodes[n]['type'] == 'CV'],
                      key=lambda x: x[1], reverse=True)[:2]

print(f"\nNombre de profils polyvalents détectés : {len(outliers)}")
for name, score, text in outliers:
    print(f"\nID du CV : {name}")
    print(f"Score de polyvalence (Betweenness) : {score:.4f}")
    print(f"Extrait du profil : {text[:150]}...")

plt.figure(figsize=(10, 6))
plt.bar([str(x[0]) for x in outliers], [x[1] for x in outliers], color='salmon')
plt.title("Top des CV à fort potentiel de polyvalence (Outliers)")
plt.ylabel("Score d'intermédiation")
plt.show()